In [ ]:
!pip install gradio

In [5]:
import tensorflow as tf
import gradio as gr
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from tensorflow.keras.models import load_model
import cv2
from ultralytics import YOLO
import os

# Initialize models as None
model = None
detection_model = None

# Path to your model
model_path = "MOH_full.keras"

# Load classification model
try:
    model = load_model(model_path)
    print("Classification model loaded successfully.")
except Exception as e:
    print(f"Error loading classification model: {e}")
    raise SystemExit("Failed to load classification model. Exiting application.")

# Load YOLO model for tumor detection
try:
    # You'll need to provide the path to your YOLO model
    # If you don't have one, you can download a pre-trained brain tumor detection model
    yolo_model_path = r"D:\ai\AI\dp\dpd\model\Tumor-Brain-Model\yoloruns\detect\brain_tumor_yolov8x\weights\best.pt"  # Update this path
    if os.path.exists(yolo_model_path):
        detection_model = YOLO(yolo_model_path)
        print("YOLO detection model loaded successfully.")
    else:
        print(f"YOLO model not found at {yolo_model_path}. Tumor detection disabled.")
        detection_model = None
except Exception as e:
    print(f"Error loading YOLO model: {e}")
    detection_model = None

# Class names for classification
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

def preprocess_image(img):
    """Preprocess image for classification model"""
    img = img.resize((256, 256))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.efficientnet_v2.preprocess_input(img_array)
    return img_array

def draw_detections(image, detections):
    """Draw bounding boxes and labels on image"""
    # Convert PIL to OpenCV format for drawing
    img_cv = np.array(image)
    img_cv = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR)
    
    colors = {
        'glioma': (255, 0, 0),      # Blue
        'meningioma': (0, 255, 0),   # Green
        'pituitary': (0, 0, 255),    # Red
        'tumor': (255, 255, 0)       # Cyan for generic tumor detection
    }
    
    for det in detections:
        x1, y1, x2, y2 = map(int, det['bbox'])
        confidence = det['confidence']
        label = det['label']
        
        # Choose color
        color = colors.get(label.lower(), (255, 0, 255))  # Magenta default
        
        # Draw bounding box
        cv2.rectangle(img_cv, (x1, y1), (x2, y2), color, 3)
        
        # Draw label background
        label_text = f"{label}: {confidence:.1f}%"
        (label_width, label_height), baseline = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(img_cv, (x1, y1 - label_height - 10), (x1 + label_width, y1), color, -1)
        
        # Draw label text
        cv2.putText(img_cv, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    # Convert back to PIL
    img_pil = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
    return img_pil

def predict_image(img):
    """Main prediction function with YOLO detection"""
    if img is None or model is None:
        return img, {"Error": "Model failed to load"}, {}, "Error"
    
    try:
        # YOLO detection
        yolo_detections = []
        yolo_has_tumor = False
        img_np = np.array(img)
        
        if detection_model:
            # Run YOLO detection
            results = detection_model(img_np, conf=0.25, verbose=False)
            
            for result in results:
                if result.boxes is not None and len(result.boxes) > 0:
                    for box in result.boxes:
                        x1, y1, x2, y2 = box.xyxy[0].tolist()
                        confidence = float(box.conf[0]) * 100
                        class_id = int(box.cls[0])
                        
                        # Get label from YOLO model's class names
                        label = detection_model.names[class_id] if hasattr(detection_model, 'names') else 'tumor'
                        
                        yolo_detections.append({
                            'bbox': [x1, y1, x2, y2],
                            'confidence': confidence,
                            'label': label
                        })
                        yolo_has_tumor = True
        
        # Draw detections on image
        if yolo_detections:
            annotated_img = draw_detections(img, yolo_detections)
        else:
            annotated_img = img
        
        # Get classification prediction
        processed_img = preprocess_image(img)
        predictions = model.predict(processed_img, verbose=0)
        
        # CRITICAL: If YOLO detects a tumor, force classification to NOT be 'notumor'
        classification_probs = {}
        if yolo_has_tumor:
            # Find the highest probability that ISN'T 'notumor'
            non_tumor_probs = [(i, p) for i, p in enumerate(predictions[0]) if class_names[i] != 'notumor']
            if non_tumor_probs:
                predicted_idx = max(non_tumor_probs, key=lambda x: x[1])[0]
            else:
                predicted_idx = np.argmax(predictions[0])
            
            # Adjust confidence message
            original_confidence = predictions[0][predicted_idx] * 100
            predicted_class = class_names[predicted_idx]
            confidence = original_confidence
            result_text = f"**Tumor Detected!**\n\n**Classification:** {predicted_class.upper()}\n**Confidence:** {confidence:.2f}%\n**Note:** YOLO detected tumor presence, overriding 'no tumor' classification."
        else:
            predicted_idx = np.argmax(predictions[0])
            predicted_class = class_names[predicted_idx]
            confidence = predictions[0][predicted_idx] * 100
            result_text = f"**Classification:** {predicted_class.upper()}\n**Confidence:** {confidence:.2f}%"
            if predicted_class == 'notumor':
                result_text += "\n**Note:** No tumor detected by either model."
            else:
                result_text += f"\n**Note:** Tumor classified as {predicted_class}"
        
        # Prepare probability dictionary
        for cls, prob in zip(class_names, predictions[0]):
            classification_probs[cls] = float(prob * 100)
        
        # Add YOLO detection info
        yolo_info = {}
        if yolo_detections:
            yolo_info['detections'] = [
                {
                    'type': det['label'],
                    'confidence': det['confidence'],
                    'bbox': det['bbox']
                } for det in yolo_detections
            ]
            if len(yolo_detections) == 1:
                result_text += f"\n\n**YOLO Detection:** {yolo_detections[0]['label']} ({yolo_detections[0]['confidence']:.1f}% confidence)"
            else:
                result_text += f"\n\n**YOLO Detections:** {len(yolo_detections)} tumors found"
        
        # Combine results
        final_results = {
            'classification': {
                'predicted_class': predicted_class,
                'confidence': confidence,
                'all_probabilities': classification_probs
            },
            'yolo_detections': yolo_info if yolo_detections else {'detections': []},
            'tumor_detected': yolo_has_tumor or predicted_class != 'notumor'
        }
        
        return annotated_img, result_text, classification_probs, predicted_class.upper()
        
    except Exception as e:
        import traceback
        error_msg = f"Prediction failed: {str(e)}\n{traceback.format_exc()}"
        print(error_msg)
        return img, f"**Error:** {str(e)}", {}, "Error"

# Create Gradio interface
with gr.Blocks(title="Brain Tumor Detection System", theme=gr.themes.Soft()) as iface:
    gr.Markdown("""
    # 🧠 Brain Tumor Detection System
    
    ### Dual-Model Analysis: YOLO + Classification
    
    Upload an MRI scan to detect and classify brain tumors using two powerful AI models:
    - **YOLO**: Detects and localizes tumors with bounding boxes
    - **(EfficientNetV2)**: Classifies tumor type (Glioma, Meningioma, Pituitary, or No Tumor)
    
    ### How it works:
    1. YOLO scans the image for tumor presence and draws bounding boxes
    2. CNN classifies the tumor type with confidence scores
    3. Results are combined for maximum accuracy
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="pil", label="Upload MRI Scan")
            upload_btn = gr.UploadButton("📁 Upload MRI Image", file_types=["image"])
            
            gr.Markdown("### 📊 Classification Results")
            result_json = gr.JSON(label="Detailed Results")
            
        with gr.Column(scale=1):
            output_image = gr.Image(type="pil", label="Detected Tumor (with Bounding Boxes)")
            
            gr.Markdown("### 🎯 Prediction")
            result_text = gr.Markdown(label="Diagnosis")
            
            gr.Markdown("### 📈 Class Probabilities")
            probabilities = gr.JSON(label="Confidence Scores")
            
            predicted_class_display = gr.Textbox(label="Final Diagnosis", interactive=False)
    
    # Example images
    gr.Markdown("### 📋 Example MRI Scans")
    gr.Examples(
        examples=[["sample_mri.jpg"]],
        inputs=[input_image],
        outputs=[output_image, result_text, probabilities, predicted_class_display],
        fn=predict_image,
        cache_examples=False,
        label="Click on an example to test"
    )
    
    # Connect components
    input_image.change(
        fn=predict_image,
        inputs=[input_image],
        outputs=[output_image, result_text, probabilities, predicted_class_display]
    )
    
    upload_btn.upload(
        fn=lambda file: Image.open(file.name),
        inputs=[upload_btn],
        outputs=[input_image]
    ).then(
        fn=predict_image,
        inputs=[input_image],
        outputs=[output_image, result_text, probabilities, predicted_class_display]
    )
    
    gr.Markdown("""
    ---
    ### ℹ️ Important Notes:
    - **Glioma**: Most common type of brain tumor
    - **Meningioma**: Usually benign tumor
    - **Pituitary**: Tumor in the pituitary gland
    - **No Tumor**: Healthy brain scan
    
    *This system is for research purposes only. Always consult medical professionals for diagnosis.*
    """)

if __name__ == "__main__":
    iface.launch(share=True)  # share=True creates public link

Classification model loaded successfully.
YOLO detection model loaded successfully.


C:\Users\engqu\AppData\Local\Temp\ipykernel_15272\1459424716.py:193: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Brain Tumor Detection System", theme=gr.themes.Soft()) as iface:


* Running on local URL:  http://127.0.0.1:7891

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
